# SQL Business Analysis

## Business Objective

The objective of this notebook is to answer business questions using SQL on the cleaned e-commerce datasets.

This notebook demonstrates the ability to:

- Load cleaned CSV datasets into a relational database
- Query multiple related tables using SQL
- Use joins, aggregations, grouping, ordering, and date-based analysis
- Translate SQL outputs into business insights
- Prepare findings for dashboard development and executive reporting

The analysis is designed to simulate how a Data Analyst would answer business questions using SQL in a real e-commerce environment.

## SQL Analysis Questions

This notebook answers the following business questions:

1. What is the total revenue generated?
2. How many orders and customers are in the database?
3. What is the average order value?
4. Which product categories generate the most revenue?
5. Which products generate the most revenue?
6. Which brands generate the most revenue?
7. How does revenue trend by month?
8. Which cities generate the most revenue?
9. Who are the highest-value customers?
10. Which products have the highest review ratings?
11. What is the event funnel?
12. Which categories have strong revenue but weaker ratings?

Each question includes a SQL query, output, and business interpretation.

## Import Libraries

In [ ]:
import pandas as pd
import sqlite3
import os
from pathlib import Path

## Set Project Directory

In [ ]:
os.chdir("/Users/saadmaher/Desktop/Data Science/Project portfolio/ecommerce-data-analysis")

PROJECT_ROOT = Path(os.getcwd())
CLEAN_DATA_DIR = PROJECT_ROOT / "data" / "cleaned"
SQL_OUTPUT_DIR = PROJECT_ROOT / "sql"

SQL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PROJECT_ROOT

## Load Cleaned Datasets

The cleaned datasets produced during the Data Cleaning phase are loaded into pandas before being written into a SQLite database.

In [ ]:
users = pd.read_csv(CLEAN_DATA_DIR / "users_clean.csv")
products = pd.read_csv(CLEAN_DATA_DIR / "products_clean.csv")
orders = pd.read_csv(CLEAN_DATA_DIR / "orders_clean.csv")
order_items = pd.read_csv(CLEAN_DATA_DIR / "order_items_clean.csv")
reviews = pd.read_csv(CLEAN_DATA_DIR / "reviews_clean.csv")
events = pd.read_csv(CLEAN_DATA_DIR / "events_clean.csv")

## Create SQLite Database

A local SQLite database is created for SQL analysis. Each cleaned dataset is written as a database table.

In [ ]:
db_path = SQL_OUTPUT_DIR / "ecommerce_analysis.db"

conn = sqlite3.connect(db_path)

users.to_sql("users", conn, if_exists="replace", index=False)
products.to_sql("products", conn, if_exists="replace", index=False)
orders.to_sql("orders", conn, if_exists="replace", index=False)
order_items.to_sql("order_items", conn, if_exists="replace", index=False)
reviews.to_sql("reviews", conn, if_exists="replace", index=False)
events.to_sql("events", conn, if_exists="replace", index=False)

db_path

## Helper Function

This function allows SQL queries to be executed and displayed as pandas DataFrames.

In [ ]:
def run_query(query):
    return pd.read_sql_query(query, conn)

# 1. Total Revenue

## Business Question

What is the total revenue generated by the business?

This is one of the most important executive KPIs.

In [ ]:
query = """
SELECT
    ROUND(SUM(item_total), 2) AS total_revenue
FROM order_items;
"""

total_revenue = run_query(query)
total_revenue

## Business Interpretation

Total revenue provides a top-level measure of business performance. This KPI will be used later in the dashboard as one of the main executive metrics.

# 2. Total Orders and Customers

## Business Question

How many orders and customers are represented in the cleaned database?

In [ ]:
query = """
SELECT
    (SELECT COUNT(DISTINCT order_id) FROM orders) AS total_orders,
    (SELECT COUNT(DISTINCT user_id) FROM users) AS total_customers,
    (SELECT COUNT(DISTINCT product_id) FROM products) AS total_products;
"""

database_size = run_query(query)
database_size

## Business Interpretation

This query summarizes the scale of the database. Total orders, customers, and products provide context for all downstream business analysis.

# 3. Average Order Value

## Business Question

What is the average order value?

Average Order Value (AOV) helps measure how much customers spend per order on average.

In [ ]:
query = """
SELECT
    ROUND(SUM(item_total) / COUNT(DISTINCT order_id), 2) AS average_order_value
FROM order_items;
"""

average_order_value = run_query(query)
average_order_value

## Business Interpretation

Average Order Value is useful for evaluating customer purchasing behavior and can help guide pricing, bundling, and promotion strategies.

# 4. Revenue by Product Category

## Business Question

Which product categories generate the most revenue?

In [ ]:
query = """
SELECT
    p.category,
    ROUND(SUM(oi.item_total), 2) AS total_revenue,
    SUM(oi.quantity) AS total_units_sold,
    COUNT(DISTINCT oi.order_id) AS total_orders
FROM order_items oi
LEFT JOIN products p
    ON oi.product_id = p.product_id
GROUP BY p.category
ORDER BY total_revenue DESC;
"""

category_revenue = run_query(query)
category_revenue

## Business Interpretation

Category-level revenue analysis identifies the strongest business segments. High-revenue categories may deserve priority in marketing, inventory planning, and dashboard reporting.

# 5. Top 10 Products by Revenue

## Business Question

Which products generate the most revenue?

In [ ]:
query = """
SELECT
    p.product_id,
    p.product_name,
    p.category,
    p.brand,
    ROUND(SUM(oi.item_total), 2) AS total_revenue,
    SUM(oi.quantity) AS total_units_sold
FROM order_items oi
LEFT JOIN products p
    ON oi.product_id = p.product_id
GROUP BY
    p.product_id,
    p.product_name,
    p.category,
    p.brand
ORDER BY total_revenue DESC
LIMIT 10;
"""

top_products = run_query(query)
top_products

## Business Interpretation

Top revenue-generating products are major business drivers. These products may be strong candidates for promotions, homepage placement, or inventory prioritization.

# 6. Revenue by Brand

## Business Question

Which brands generate the most revenue?

In [ ]:
query = """
SELECT
    p.brand,
    ROUND(SUM(oi.item_total), 2) AS total_revenue,
    SUM(oi.quantity) AS total_units_sold,
    COUNT(DISTINCT oi.order_id) AS total_orders
FROM order_items oi
LEFT JOIN products p
    ON oi.product_id = p.product_id
GROUP BY p.brand
ORDER BY total_revenue DESC
LIMIT 10;
"""

brand_revenue = run_query(query)
brand_revenue

## Business Interpretation

Brand-level revenue analysis can support supplier negotiations, brand partnerships, and merchandising strategy.

# 7. Monthly Revenue Trend

## Business Question

How does revenue change over time?

In [ ]:
query = """
SELECT
    SUBSTR(o.order_date, 1, 7) AS order_month,
    ROUND(SUM(oi.item_total), 2) AS total_revenue,
    COUNT(DISTINCT o.order_id) AS total_orders
FROM orders o
LEFT JOIN order_items oi
    ON o.order_id = oi.order_id
GROUP BY SUBSTR(o.order_date, 1, 7)
ORDER BY order_month;
"""

monthly_revenue = run_query(query)
monthly_revenue

## Business Interpretation

Monthly revenue trends help identify seasonality, growth patterns, or periods that require deeper investigation.

# 8. Revenue by City

## Business Question

Which cities generate the most revenue?

In [ ]:
query = """
SELECT
    u.city,
    ROUND(SUM(oi.item_total), 2) AS total_revenue,
    COUNT(DISTINCT o.order_id) AS total_orders,
    COUNT(DISTINCT u.user_id) AS unique_customers
FROM orders o
LEFT JOIN users u
    ON o.user_id = u.user_id
LEFT JOIN order_items oi
    ON o.order_id = oi.order_id
GROUP BY u.city
ORDER BY total_revenue DESC
LIMIT 10;
"""

city_revenue = run_query(query)
city_revenue

## Business Interpretation

City-level revenue analysis helps identify strong geographic markets. These insights can support regional marketing campaigns and logistics decisions.

# 9. Highest-Value Customers

## Business Question

Which customers generate the most revenue?

In [ ]:
query = """
SELECT
    u.user_id,
    u.name,
    u.city,
    ROUND(SUM(oi.item_total), 2) AS customer_revenue,
    COUNT(DISTINCT o.order_id) AS total_orders
FROM users u
LEFT JOIN orders o
    ON u.user_id = o.user_id
LEFT JOIN order_items oi
    ON o.order_id = oi.order_id
GROUP BY
    u.user_id,
    u.name,
    u.city
ORDER BY customer_revenue DESC
LIMIT 10;
"""

top_customers = run_query(query)
top_customers

## Business Interpretation

High-value customers are important for retention strategy. These customers may be good candidates for loyalty programs, personalized offers, or VIP segmentation.

# 10. Highest Rated Products

## Business Question

Which products have the highest average review ratings?

In [ ]:
query = """
SELECT
    p.product_id,
    p.product_name,
    p.category,
    ROUND(AVG(r.rating), 2) AS average_review_rating,
    COUNT(r.review_id) AS review_count
FROM reviews r
LEFT JOIN products p
    ON r.product_id = p.product_id
GROUP BY
    p.product_id,
    p.product_name,
    p.category
HAVING review_count >= 5
ORDER BY average_review_rating DESC, review_count DESC
LIMIT 10;
"""

highest_rated_products = run_query(query)
highest_rated_products

## Business Interpretation

Products with strong ratings and sufficient review volume may indicate high customer satisfaction. These products can be highlighted in marketing or used as benchmarks for quality.

# 11. Event Funnel

## Business Question

How are users moving through the behavioral funnel?

In [ ]:
query = """
SELECT
    event_type,
    COUNT(*) AS event_count,
    COUNT(DISTINCT user_id) AS unique_users
FROM events
GROUP BY event_type
ORDER BY event_count DESC;
"""

event_funnel = run_query(query)
event_funnel

## Business Interpretation

The event funnel shows how users interact with the platform. Comparing views, carts, wishlists, and purchases helps identify potential conversion opportunities or friction points.

# 12. Revenue and Rating by Category

## Business Question

Which categories generate strong revenue but may have weaker customer ratings?

In [ ]:
query = """
SELECT
    p.category,
    ROUND(SUM(oi.item_total), 2) AS total_revenue,
    ROUND(AVG(r.rating), 2) AS average_review_rating,
    COUNT(DISTINCT r.review_id) AS review_count
FROM products p
LEFT JOIN order_items oi
    ON p.product_id = oi.product_id
LEFT JOIN reviews r
    ON p.product_id = r.product_id
GROUP BY p.category
ORDER BY total_revenue DESC;
"""

category_revenue_rating = run_query(query)
category_revenue_rating

## Business Interpretation

This analysis compares commercial performance with customer satisfaction. Categories with high revenue but weaker ratings may require quality review, better product descriptions, or customer experience improvements.

# Export SQL Query Results

The main SQL outputs are exported as CSV files so they can be reused in reporting or dashboard development.

In [ ]:
SQL_RESULTS_DIR = SQL_OUTPUT_DIR / "query_results"
SQL_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

category_revenue.to_csv(SQL_RESULTS_DIR / "category_revenue.csv", index=False)
top_products.to_csv(SQL_RESULTS_DIR / "top_products.csv", index=False)
brand_revenue.to_csv(SQL_RESULTS_DIR / "brand_revenue.csv", index=False)
monthly_revenue.to_csv(SQL_RESULTS_DIR / "monthly_revenue.csv", index=False)
city_revenue.to_csv(SQL_RESULTS_DIR / "city_revenue.csv", index=False)
top_customers.to_csv(SQL_RESULTS_DIR / "top_customers.csv", index=False)
event_funnel.to_csv(SQL_RESULTS_DIR / "event_funnel.csv", index=False)

list(SQL_RESULTS_DIR.iterdir())

# Executive Summary

## Objective

The objective of this notebook was to answer key e-commerce business questions using SQL.

## SQL Skills Demonstrated

This notebook demonstrates:

- Aggregations using `SUM`, `COUNT`, and `AVG`
- Grouping with `GROUP BY`
- Sorting with `ORDER BY`
- Filtering grouped results with `HAVING`
- Joining multiple relational tables
- Date-based analysis using month extraction
- Exporting query results for reporting

## Key Business Areas Analyzed

The SQL analysis covered:

- Revenue performance
- Order and customer volume
- Average order value
- Category performance
- Product performance
- Brand performance
- Monthly revenue trends
- Geographic revenue
- High-value customers
- Product ratings
- Behavioral event funnel

## Business Value

The SQL analysis translates cleaned and integrated e-commerce data into business insights. These results can support decision-making in marketing, product strategy, inventory planning, customer retention, and dashboard development.

## Next Steps

The next stage of the project is Power BI dashboard development. The KPIs and SQL outputs from this notebook will guide dashboard structure, visual selection, and executive reporting priorities.

## Close Database Connection

In [ ]:
conn.close()